# **DeepLIFT further: RevealCancel and DeepSHAP**

A practice for the module ["Attribution from axioms"](https://open-xai-platform.web.app).

The lesson covered two continuations of DeepLIFT and left both at the level of words. Here we
compute them.

**RevealCancel.** The lesson gives an exact example for it — the neuron
$h=\mathrm{ReLU}(i-j)$ with the reference $i^0=j^0=0$ and the input $i=j=5$ — and explains in
words why Rescale is blind there. The example takes five lines to compute, and a computation
convinces more than an explanation.

**DeepSHAP.** The lesson says: average DeepLIFT over references from the data and you get an
approximation of Shapley values. Let us check what happens to the sum of the attributions.

It runs instantly.

In [ ]:
import torch
import torch.nn as nn

def g(v):
    """A ReLU for a single number — both rules will need it."""
    return max(0.0, float(v))


def rescale(z0, dz_pos, dz_neg):
    """Rescale: a single secant over the total deviation of the input."""
    dz = dz_pos + dz_neg
    if abs(dz) < 1e-9:
        return 0.0, 0.0                 # the degenerate case: the deviation of the input is zero
    m = (g(z0 + dz) - g(z0)) / dz
    return m * dz_pos, m * dz_neg


def reveal_cancel(z0, dz_pos, dz_neg):
    """RevealCancel: the positive and negative parts pass through the nonlinearity separately."""
    dg_pos = 0.5 * ((g(z0 + dz_pos) - g(z0))
                    + (g(z0 + dz_neg + dz_pos) - g(z0 + dz_neg)))   # the positive part: first alone, then after the negative one
    dg_neg = 0.5 * ((g(z0 + dz_neg) - g(z0))
                    + (g(z0 + dz_pos + dz_neg) - g(z0 + dz_pos)))   # the negative part: mirrored
    return dg_pos, dg_neg


def show(title, z0, dz_pos, dz_neg):
    dh = g(z0 + dz_pos + dz_neg) - g(z0)
    print(f'{title}   deviation of the output {dh:+.2f}')
    for name, (a, b) in (('Rescale', rescale(z0, dz_pos, dz_neg)),
                         ('RevealCancel', reveal_cancel(z0, dz_pos, dz_neg))):
        print(f'   {name:14} i {float(a):+.2f}   j {float(b):+.2f}   sum {float(a)+float(b):+.2f}')

## 1. Complete cancellation: the very place where Rescale is blind

The input $i$ pushes the neuron up by 5, the input $j$ pushes it down by 5. The output has not
moved at all.

In [ ]:
show(f'Complete cancellation: i = 5, j = 5', z0=0.0, dz_pos=5.0, dz_neg=-5.0)

**Rescale gave zero to both.** And that is not a bug: it honestly reports that the
output did not change. But it hid the entire mechanics of what happened — from its map you
cannot tell "neither input matters" from "both matter and cancelled each other".

**RevealCancel gave $+2.50$ and $-2.50$.** It passed the positive and negative parts through the
nonlinearity separately and saw what was going on.

Note the sums: **zero for both**. Neither rule violates summation-to-delta — the same sum is
simply decomposed differently. That matters: RevealCancel does not fix a broken axiom, it
reveals what the axiom permits.

**Task 1.** Check what happens if the roles are swapped: $i=-5$, $j=-5$, so that
$\Delta z^{+}=5$ becomes $\Delta z^{-}$ and vice versa. Is RevealCancel symmetric?

In [ ]:
# Your code here

## 2. And if the cancellation is partial

The lesson says that with $j=4$ "the neuron comes alive instantly". Let us see what both rules
say.

In [ ]:
show(f'Partial cancellation: i = 5, j = 4', z0=0.0, dz_pos=5.0, dz_neg=-4.0)

This is worth looking at closely — **the lesson does not talk about it at all.**

Here Rescale is no longer blind: the output moved by 1, and the rule decomposed that shift as
$+5$ and $-4$ — that is, it attributed to the inputs their full deviations. RevealCancel gave
$+3$ and $-2$.

**Both decompositions sum to the same $+1$**, and both are legitimate. But they say different
things. Rescale: "$i$ pushed by 5, $j$ brought it back by 4". RevealCancel averages over the two
orders of addition and is therefore gentler: it assigns part of the effect of each input to
their joint action rather than to each of them alone.

The conclusion worth taking away: **the rules diverge not only in the degenerate case.** The
choice between them is substantive, not an emergency measure, and it has to be declared always,
not only when something has broken.

**Task 2.** Build both decompositions for $j$ from 0 to 5 in steps of 1 and plot the two curves
of the contribution of $i$. Where do they diverge most, and why exactly there?

In [ ]:
# Your code here

## 3. DeepSHAP: the same device as in Expected Gradients

The lesson points out: Expected Gradients relate to Integrated Gradients exactly as DeepSHAP
relates to DeepLIFT. A single fixed reference is replaced by a distribution of references.

First let us compute DeepLIFT with one zero reference, as in the earlier lessons.

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 6, bias=False), nn.ReLU(),
                    nn.Linear(6, 3, bias=False)).eval()
TARGET = 1


def trace(x):
    acts = [x]
    for layer in net:
        x = layer(x)
        acts.append(x)
    return acts


def deeplift(x, baseline, c=TARGET):
    """DeepLIFT (Rescale) on a network: multipliers, the chain rule, the input deviation."""
    a_x, a_0 = trace(x), trace(baseline)
    m = torch.zeros_like(a_x[-1])
    m[0, c] = 1.0
    for i in range(len(net) - 1, -1, -1):
        layer = net[i]
        if isinstance(layer, nn.Linear):
            m = m @ layer.weight
        else:
            dz, dx = a_x[i + 1] - a_0[i + 1], a_x[i] - a_0[i]
            safe = torch.where(dx.abs() > 1e-7, dx, torch.ones_like(dx))
            m = m * torch.where(dx.abs() > 1e-7, dz / safe, torch.zeros_like(dx))
    return (m * (x - baseline)).detach()


torch.manual_seed(1)
data = torch.rand(200, 4)          # the training set, which doubles as the distribution of references
x = torch.rand(1, 4) + 0.3

single = deeplift(x, torch.zeros(1, 4))
print(f'a single zero reference: {single.numpy().round(4)}   sum {single.sum():+.4f}')

Now let us replace the single reference with a distribution: take references from
the data and average the attributions over them.

In [ ]:
for n in (1, 5, 25, 200):
    attributions = torch.stack([deeplift(x, data[i:i + 1]) for i in range(n)]).mean(0)
    mean_out = torch.stack([net(data[i:i + 1])[0, TARGET] for i in range(n)]).mean()
    delta = (net(x)[0, TARGET] - mean_out).item()
    print(f'averaged over references, count {n:3}: {attributions.numpy().round(4)}   sum {attributions.sum():+.4f}   '
          f'f(x) - mean f(reference): {delta:+.4f}')

Look at the last two columns of each row: **the sum of the attributions matches the
difference between the output on the object and the mean output on the references — for every
number of references, to four decimals.**

That is summation-to-delta, only $\Delta t$ is now measured not from a single point but from
the mean over the distribution. The property survived the replacement of the reference by a
distribution, and that is the whole point of the device: we removed the arbitrariness of
choosing a point without losing any guarantee.

You can also see the attributions settle down as the number of references grows: from one to
five they change noticeably, from twenty-five to two hundred hardly at all.

**Task 3.** Compare the DeepSHAP attributions with the Expected Gradients attributions on the
same object and the same set of references (the Expected Gradients code is in the "Expected
Gradients" notebook of this same module). How close are they? The lesson claims both approximate
the same thing — check it.

In [ ]:
# Your code here

## What to take away from this notebook

- **Rescale is blind to mutual cancellation, and it shows in a number.** Under complete
  cancellation it gives zero to both inputs, RevealCancel gives $+2.5$ and $-2.5$. Both
  decompositions sum to zero: neither rule violates the axiom, the difference is that one rule
  shows the mechanics and the other does not.
- **The rules diverge under partial cancellation too.** The choice between Rescale and
  RevealCancel is a substantive decision that must be declared next to the map, not a silent
  implementation detail.
- **DeepSHAP preserves summation-to-delta**, only the deviation is now measured from the mean
  over the distribution of references. Replacing a point with a distribution removes the
  arbitrariness without losing the guarantee.
- **The device is the same in both families:** Expected Gradients relate to Integrated Gradients
  as DeepSHAP relates to DeepLIFT.